In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_distances
from joblib import Parallel, delayed

# --- Setup Imports ---
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "evaluation").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from evaluation.split import leave_last_out_split
from models.baselines import PopularityRecommender
from models.mf import BPRMatrixFactorization
from models.content_based import ContentBasedRecommender, HybridMFContentRecommender

print("=== 1. PREPARING DATA ===")
interactions = pd.read_csv("../data/raw/RAW_interactions.csv")
recipes = pd.read_csv("../data/raw/RAW_recipes.csv")

interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")
positive_df = interactions[interactions["rating"] >= 4].copy().rename(columns={"date": "date_time"})
user_counts = positive_df["user_id"].value_counts()
positive_df = positive_df[positive_df["user_id"].isin(user_counts[user_counts >= 2].index)].copy()

recipes_model = recipes.rename(columns={"id": "recipe_id"}).copy()
df_model = positive_df.merge(recipes_model, on="recipe_id", how="inner")

train_df, test_df = leave_last_out_split(df_model, user_col="user_id", time_col="date_time")
train_history = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

# Wir evaluieren auf 5000 Usern für ein robustes Ergebnis
rng = np.random.default_rng(42)
audit_users = rng.choice(test_df["user_id"].unique(), size=5000, replace=False)

print("\n=== 2. TRAINING MODELS (POPULARITY, BPR, CONTENT) ===")
# Popularity
pop_model = PopularityRecommender()
pop_model.fit(train_df, item_col="recipe_id")

# BPR (3 epochs is enough to get the structural popularity bias for the audit)
bpr_model = BPRMatrixFactorization(k_factors=32, epochs=3)
bpr_model.fit(train_df, item_col="recipe_id")

# Content
content_model = ContentBasedRecommender(
    text_cols=["name", "tags", "ingredients", "description"],
    metadata_cols=["minutes", "n_steps", "n_ingredients"],
    min_df=2, ngram_range=(1, 1), time_col="date_time"
)
content_model.fit(train_df, recipes_model, user_col="user_id", item_col="recipe_id")

# Hybrid
hybrid_model = HybridMFContentRecommender(bpr_model, content_model, alpha=0.9, adaptive=True)

print("\n=== 3. GENERATING RECOMMENDATIONS ===")
def get_recs(user):
    hist = train_history.get(user, set())
    return {
        "pop": pop_model.recommend(user, hist, k=20),
        "hybrid": hybrid_model.recommend(user, hist, k=20)
    }

results = Parallel(n_jobs=-1, batch_size=100)(delayed(get_recs)(u) for u in audit_users)

pop_all_recs = [r["pop"] for r in results]
hybrid_all_recs = [r["hybrid"] for r in results]

print("\n=== 4. CALCULATING ACTUAL GINI & ILD ===")

# --- GINI COEFFICIENT ---
def calculate_gini_correct(lists, total_items=206817):
    all_items = [item for sublist in lists for item in sublist]
    counts = pd.Series(all_items).value_counts()

    # HIER IST DER FIX: Wir füllen alle fehlenden Items mit 0 auf!
    # (Wir simulieren ein Array der Länge 206817)
    counts_array = np.zeros(total_items)
    counts_array[:len(counts)] = counts.values

    counts_array = np.sort(counts_array)
    index = np.arange(1, total_items + 1)

    return ((np.sum((2 * index - total_items  - 1) * counts_array)) / (total_items * np.sum(counts_array)))

# Wenn du das aufrufst:
print(f"Gini - Popularity:    {calculate_gini_correct(pop_all_recs):.4f}")
print(f"Gini - Hybrid Model:  {calculate_gini_correct(hybrid_all_recs):.4f}")


# --- INTRA-LIST DIVERSITY (ILD) ---
def calculate_ild(lists, content_model):
    ild_scores = []
    # Fetch the TF-IDF matrix from the content model
    feature_matrix = content_model.item_feature_matrix
    mapping = content_model.item_mapping

    for rec_list in lists:
        # Get matrix indices for the recommended items
        valid_items = [mapping[item] for item in rec_list if item in mapping]
        if len(valid_items) < 2:
            continue

        # Get vectors and compute pairwise cosine distances (1 - cosine_similarity)
        vectors = feature_matrix[valid_items]
        distances = cosine_distances(vectors)

        # Average distance of upper triangle (excluding self-distance)
        n = distances.shape[0]
        mean_dist = np.sum(np.triu(distances, k=1)) / ((n * (n - 1)) / 2)
        ild_scores.append(mean_dist)

    return np.mean(ild_scores)

pop_ild = calculate_ild(pop_all_recs, content_model)
hybrid_ild = calculate_ild(hybrid_all_recs, content_model)

print(f"ILD  - Popularity:    {pop_ild:.4f}")
print(f"ILD  - Hybrid Model:  {hybrid_ild:.4f}")
print("\nDone! Use these EXACT numbers for the report.")

=== 1. PREPARING DATA ===
Sorting data chronologically to prevent time leakage...
Dropping users with tied final timestamps because a strict leave-last-out split is not identifiable: 8115 users
Splitting data: Leave-last-out...
Train Set: 707232 interactions
Test Set: 44249 interactions

=== 2. TRAINING MODELS (POPULARITY, BPR, CONTENT) ===
Training Popularity Baseline...
Learned popularity ranking for 185226 unique recipes.
Training BPR MF (k=32, epochs=3)...
Epoch 1/3 completed.
Epoch 2/3 completed.
Epoch 3/3 completed.
BPR Training complete!
Training Content-Based Recommender...
Using text columns: ['name', 'tags', 'ingredients', 'description']
Using metadata columns: ['minutes', 'n_steps', 'n_ingredients']
Feature dimension: 45344
Vocabulary size: 45344
Missing rates by feature: {'name': 0.0, 'tags': 0.0, 'ingredients': 0.0, 'description': 0.021638430889831882, 'minutes': 0.0, 'n_steps': 0.0, 'n_ingredients': 0.0}
Content model trained with 185226 items and 44249 user profiles.

==